In [1]:
from raven import raven
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import openml

In [ ]:
# Load and Prepare Data
dataset_id = 1457
print(f"Loading OpenML dataset {dataset_id}...")

dataset = openml.datasets.get_dataset(dataset_id)

# Get data, separating features (X) and target (y)
X, y, _, _ = dataset.get_data(
    dataset_format="dataframe", 
    target=dataset.default_target_attribute
)


X = X.select_dtypes(include=[np.number])
# Fill missing values
X = X.fillna(0) 

print(f"Data loaded: {X.shape[0]} samples, {X.shape[1]} numeric features.")
print(f"Target column: {y.name}")

Loading OpenML dataset 1457...
Data loaded: 1500 samples, 10000 numeric features.
Target column: Class


In [ ]:
essential_features, redundant_features = raven(
    data = X,
    mode='df',
    sample_size=50,
    tau=0.95
)

# Corrected Print Statement 
# The total features processed is the sum of essential and redundant ones
total_numeric_features = len(essential_features) + len(redundant_features)

print(f"{len(redundant_features)} redundant columns identified, out of {total_numeric_features} total numeric features.")

KeyboardInterrupt: 

In [47]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

def train(x, y, reduction_status = ""):
    train_size = int(len(x) * 0.8)

    x_train = x.iloc[:train_size, :]
    y_train = y.iloc[:train_size]
    x_test = x.iloc[train_size:, :]
    y_test = y.iloc[train_size:]

    model = RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42)

    model.fit(x_train, y_train)

    y_pred = model.predict(x_test)

    mse_without_reduction = mean_squared_error(y_test, y_pred)
    r2_without_reduction = r2_score(y_test, y_pred)
    print(f"Mean Squared Error{reduction_status}: ", mse_without_reduction)
    print(f"R2 Score{reduction_status}: ", r2_without_reduction)
train(X, y, " without reduction")

Mean Squared Error without reduction:  105876431.65797633
R2 Score without reduction:  -75.85618026218481


In [29]:
x1 = X[[c for c in essential_features if c in X.columns]]
train(x1, y, " with reduction")


Mean Squared Error with reduction:  1.739350041391505e-11
R2 Score with reduction:  0.9792278623335091


RAVEN ON QSAR datasets with 1000 features

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
from scipy.sparse import issparse
from raven import raven


🚀 Starting RAVEN + Random Forest in [CSV] mode...
   Loading CSV: datasets/df_destx.csv...
   ℹ️ Target not set. Auto-selected last column: 'RightDestxS_temporal_transverseCurvInd'
   ℹ️ Dropped 9 non-numeric features.
   Loaded: 167 samples, 716 features.
   Preparing data for RAVEN...
   Running RAVEN (Tau=0.99, Sample=1000)...
Building redundancy graph...
Identifying essential and redundant features...
   ✅ RAVEN Selected: 642 features
   🗑️ RAVEN Dropped:  74 features
   Training Random Forest...

📊 MODEL PERFORMANCE (RAVEN + RF)
R² Score:         0.6151
MAE Error:        0.0379
MSE Error:        0.0021
----------------------------------------
Original Features: 716
Used Features:     642
Compression:       10.3%


In [ ]:

SOURCE_TYPE = 'CSV'   

OPENML_ID = 3153      

CSV_PATH = "datasets/df_destx.csv"
TARGET_COL = None     # Auto-detect if None

RAVEN_SAMPLE = 1000  
RAVEN_TAU = 0.99



In [ ]:
print(f"Starting RAVEN + Random Forest in [{SOURCE_TYPE}] mode...")

X_df = None
y = None

if SOURCE_TYPE == 'OPENML':
    print(f"Fetching OpenML ID {OPENML_ID}")
    try:
        dataset = fetch_openml(data_id=OPENML_ID, as_frame=True, parser='auto')
        X_df = dataset.data
        y = dataset.target
        
        # Ensure y is numeric
        if not np.issubdtype(y.dtype, np.number):
             y = LabelEncoder().fit_transform(y)
             
        print(f" Loaded: {X_df.shape[0]} samples, {X_df.shape[1]} features.")
        
    except Exception as e:
        print(f" Error loading OpenML: {e}")
        exit()

elif SOURCE_TYPE == 'CSV':
    print(f" Loading CSV: {CSV_PATH}")
    try:
        df = pd.read_csv(CSV_PATH)
        
        # Auto-detect target
        if TARGET_COL is None:
            TARGET_COL = df.columns[-1]
            print(f"Target not set. Auto-selected last column: '{TARGET_COL}'")
        
        if TARGET_COL not in df.columns:
            raise ValueError(f"Target column '{TARGET_COL}' not found.")
            
        y = df[TARGET_COL].values
        X_df = df.drop(columns=[TARGET_COL])
        
        # Handle Categorical Target
        if not np.issubdtype(y.dtype, np.number):
            print(" Target is text. Encoding to numbers")
            y = LabelEncoder().fit_transform(y)
            
        # Drop non-numeric features for RAVEN
        X_numeric = X_df.select_dtypes(include=[np.number])
        if X_numeric.shape[1] < X_df.shape[1]:
            print(f" Dropped {X_df.shape[1] - X_numeric.shape[1]} non-numeric features.")
        X_df = X_numeric

        print(f"   Loaded: {X_df.shape[0]} samples, {X_df.shape[1]} features.")
        
    except Exception as e:
        print(f"Error loading CSV: {e}")
        exit()


In [ ]:

# RAVEN requires a Dense DataFrame to calculate correlations.
# If OpenML returned Sparse data, we densify it.

print(" Preparing data for RAVEN")

# fillna is important before correlation check
imputer = SimpleImputer(strategy='mean')
X_dense = pd.DataFrame(imputer.fit_transform(X_df), columns=X_df.columns)


In [ ]:

print(f"   Running RAVEN (Tau={RAVEN_TAU}, Sample={RAVEN_SAMPLE})...")

essential_feats, redundant_feats = raven(
    mode = 'df',
    data=X_dense,
    sample_size=RAVEN_SAMPLE,
    tau=RAVEN_TAU
)

# Apply Filter
X_raven = X_dense[essential_feats]

print(f" RAVEN Selected: {len(essential_feats)} features")
print(f" RAVEN Dropped:  {len(redundant_feats)} features")


In [ ]:

print("   Training Random Forest")

X_train, X_test, y_train, y_test = train_test_split(
    X_raven, y, test_size=0.2, random_state=42
)

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1  # Use all cores
    ))
])

rf_pipeline.fit(X_train, y_train)


In [ ]:
y_pred = rf_pipeline.predict(X_test)

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
print(f"R² Score:         {r2:.4f}")
print(f"MAE Error:        {mae:.4f}")
print(f"MSE Error:        {mse:.4f}")

print(f"Original Features: {X_df.shape[1]}")
print(f"Used Features:     {len(essential_feats)}")
print(f"Compression:       {100 - (len(essential_feats)/X_df.shape[1]*100):.1f}%")


RAVEN ON LIMESODA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline

SOURCE_TYPE = 'CSV' 
OPENML_ID = 42572
CSV_PATH = "datasets/df_destx.csv" 
TARGET_COL = None                  


🚀 Starting LASSO Pipeline in [CSV] mode...
   Loading CSV: datasets/df_destx.csv...
   ℹ️ Target not set. Auto-selected last column: 'RightDestxS_temporal_transverseCurvInd'
   ℹ️ Dropped 9 non-numeric columns.
   Loaded: 167 samples, 716 features.
   Splitting Train/Test...
   Training LASSO Model (Cross-Validation)...
   Evaluating...

📊 FINAL RESULTS (CSV)
R² Score:          0.6217
MAE Error:         0.0378
MSE Error:         0.0022
----------------------------------------
Total Features:    716
Selected Features: 22
Sparsity:          96.9% features dropped
Best Alpha:        0.005893
✅ Model fits the data well.


In [ ]:

print(f" Starting LASSO Pipeline in [{SOURCE_TYPE}] mode...")

X, y = None, None
feature_names = None

if SOURCE_TYPE == 'OPENML':
    print(f"   Fetching OpenML ID {OPENML_ID}...")
    dataset = fetch_openml(data_id=OPENML_ID, as_frame=True, parser='auto')
    
    # Get Dataframes
    X = dataset.data
    y = dataset.target
    
    # Handle if target is categorical (convert to numeric for Regression)
    if y.dtype == 'object' or isinstance(y.dtype, pd.CategoricalDtype):
        print("    Target is categorical. Encoding to numeric...")
        y = LabelEncoder().fit_transform(y)
        
    feature_names = np.array(X.columns)
    print(f"   Loaded: {X.shape[0]} samples, {X.shape[1]} features.")

elif SOURCE_TYPE == 'CSV':
    print(f"   Loading CSV: {CSV_PATH}...")
    df = pd.read_csv(CSV_PATH)
    
    # Check if Target Exists
    if TARGET_COL is None:
        TARGET_COL = df.columns[-1]  # Auto-select last column as target
        print(f"Target not set. Auto-selected last column: '{TARGET_COL}'")

    # Separate X and y
    y = df[TARGET_COL].values
    X = df.drop(columns=[TARGET_COL])
    
    # Drop Non-Numeric Columns (IDs, Dates, etc.)
    X_numeric = X.select_dtypes(include=[np.number])
    dropped = [c for c in X.columns if c not in X_numeric.columns]
    if dropped:
        print(f" Dropped {len(dropped)} non-numeric columns.")
    
    X = X_numeric
    feature_names = np.array(X.columns)
    print(f" Loaded: {X.shape[0]} samples, {X.shape[1]} features.")



In [ ]:

print("Splitting Train/Test")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("Training LASSO Model (Cross-Validation)")
alpha_val = 100  # Range of alpha values to test
# Pipeline handles Imputation -> Scaling -> Modeling automatically
lasso_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),     # Fill missing values
    ('scaler', StandardScaler(with_mean=False)),     # Scale data (Required for LASSO)
    ('lasso', LassoCV(alphas=alpha_val,cv=5, random_state=42, max_iter=10000, n_jobs=-1)) # Auto-tune Alpha
])
lasso_pipeline.fit(X_train, y_train)

In [ ]:

print("Evaluating")
y_pred = lasso_pipeline.predict(X_test)

# Metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

# Extract Model Details
lasso_model = lasso_pipeline.named_steps['lasso']
n_features = X_train.shape[1]
n_selected = np.sum(lasso_model.coef_ != 0)
sparsity = (1 - (n_selected / n_features)) * 100


In [ ]:
print(f" FINAL RESULTS ({SOURCE_TYPE})")
print(f"R² Score:          {r2:.4f}")
print(f"MAE Error:         {mae:.4f}")
print(f"MSE Error:         {mse:.4f}")
print(f"Total Features:    {n_features}")
print(f"Selected Features: {n_selected}")
print(f"Sparsity:          {sparsity:.1f}% features dropped")
print(f"Best Alpha:        {lasso_model.alpha_:.6f}")